In [8]:
from pymatgen.core import Structure
from pymatgen.ext.matproj import MPRester
from ase.io import write
from ase.io.extxyz import write_extxyz
from pymatgen.io.ase import AseAtomsAdaptor
import logging
import numpy as np

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("MP_collector")

def supercell_creation(atoms, min_rcutoff=4.0):
    """Create a supercell by replicating the unit cell.

    Args:
    atoms (ASE Atoms): The original ASE Atoms object.
    min_rcutoff (float): The minimum cutoff radius to 
        be ensured in the supercell.
    
    Returns:
    ASE Atoms: The replicated supercell ASE Atoms object.
    """
    min_rcutoff = np.array([min_rcutoff]*3)
    min_lengths = np.array(atoms.get_cell().lengths())
    replication_factors = tuple(np.ceil(min_rcutoff / min_lengths).astype(int))
    print("Replication factors:", replication_factors)
    atoms = atoms.repeat(replication_factors)
    
    return atoms

def random_displacement(atoms, epsilon=0.05):
    """Apply a random displacement to each atom in the ASE Atoms object.

    Args:
    atoms (ASE Atoms): The original ASE Atoms object.
    epsilon (float): The maximum random displacement in Angstrom.
    
    Returns:
    ASE Atoms: The displaced ASE Atoms object.
    """
    displacement = np.random.uniform(-epsilon, epsilon, size=(len(atoms), 3))
    atoms.positions += displacement
    return atoms

def random_scaling(atoms, scaling_range=(0.95, 1.05)):
    """Apply a random scaling to the volume of the ASE Atoms object.

    Args:
    atoms (ASE Atoms): The original ASE Atoms object.
    scaling_range (tuple): The range for random scaling factors.

    Returns:
    ASE Atoms: The scaled ASE Atoms object.
    """
    scaling_factor = np.random.uniform(scaling_range[0], scaling_range[1])
    atoms.set_cell(atoms.get_cell() * scaling_factor, scale_atoms=True)
    return atoms


# Material Project API key
API_KEY = "pkVzTevpTWptwd33e5HA1UmiN3Xt0tp1"

# Parameters for data-augmentation
epsilon = 0.05  # Maximum random displacement in Angstrom
scaling_range = (0.95, 1.05)  # Volume scaling range

# Converter between pymatgen Structure and ASE Atoms
adaptor = AseAtomsAdaptor()

# System to search
system = "Cu"  

num_structures = 50

atoms_list = []

with MPRester(API_KEY) as mpr:
    
    # Download of the structures
    list_of_structures = mpr.get_structures(system)
    num_augmented = num_structures // len(list_of_structures) 
    
    for struct in list_of_structures:

        # Convert pymatgen Structure to ASE Atoms
        atoms = adaptor.get_atoms(struct)
        atoms = supercell_creation(atoms, min_rcutoff=4.0)
        
        for _ in range(num_augmented):  # Number of augmentations per structure
            aug_atoms = atoms.copy()
            aug_atoms = random_displacement(aug_atoms, epsilon=epsilon)
            aug_atoms = random_scaling(aug_atoms, scaling_range=scaling_range)
            atoms_list.append(aug_atoms)

        atoms_list.append(atoms)

write(f"MP_{system}.extxyz", atoms_list, format="extxyz")

print(f"Scritte {len(atoms_list)} strutture in 'MP_{system}.extxyz'")


Replication factors: (2, 1, 1)
Replication factors: (2, 2, 2)
Replication factors: (2, 2, 1)
Replication factors: (2, 2, 1)
Replication factors: (2, 2, 1)
Replication factors: (2, 2, 2)
Replication factors: (2, 2, 2)
Replication factors: (2, 2, 1)
Scritte 56 strutture in 'MP_Cu.extxyz'
